# Configuration

Set the project root directory below (the root of the git repository that was cloned).

In [1]:
PROJECT_ROOT_DIR = 'C:/Users/sumit/GitRepos/FloodResearch/bhccpx/'

import os
import sys
os.chdir(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))
sys.path.append(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))

Place this same directory in DEFAULT:rootdir in the `bhccpx/bhc_complex.ini` configuration file.

In [2]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import zipfile
import pickle as pkl

import bhc_datautil
import csv2sys
import sys2bhc

In [3]:
config = bhc_datautil.read_config()

# Downloading Data

Go to the [FFIC NIC Data Download](https://www.ffiec.gov/npw/FinancialReport/DataDownload) site and download all of the CSV ZIP files. This step cannot be automated, as it may require a CAPTCHA.

Once you have done so, place them in the folder you specified in the DEFAULT::data directory. You can either unzip them manually (and place the CSV files in the same directory) or run the following utility.

Set IDs, and set asofdates.

In [9]:
print(f"Data directory: {config.get('DEFAULT', 'datadir')}")

filenames = [
	'CSV_ATTRIBUTES_ACTIVE.zip',
	'CSV_ATTRIBUTES_BRANCHES.zip',
	'CSV_ATTRIBUTES_CLOSED.zip',
	'CSV_RELATIONSHIPS.zip',
	'CSV_TRANSFORMATIONS.zip',
]
for filename in filenames:
	zip_path = os.path.join(config.get('DEFAULT', 'datadir'), filename)
	if os.path.exists(zip_path):
		with zipfile.ZipFile(zip_path, 'r') as zip_ref:
			zip_ref.extractall(config.get('DEFAULT', 'datadir'))
		print(f"Extracted {zip_path}")

Data directory: C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_ACTIVE.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_BRANCHES.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_CLOSED.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_RELATIONSHIPS.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_TRANSFORMATIONS.zip


# Parsing the data
We next parse the raw CSV data into NetworkX objects, which are cached locally.  Parsing is an expensive operation, so it is highly desirable to cache the NetworkX objects as intermediate state.

This operation may take some time to complete. If you have a strong CPU with multiple cores, we recommend adjusting the DEFAULT:parallelcores option in the config file to take advantage of parallel processing.

In [ ]:
csv2sys.main(argv=['csv2sys.py'])

In [ ]:
sys2bhc.main(argv=['sys2bhc.py'])

# Extracting a time series
As an example, we will track **SVB Financial Group** (RSSD=1031449) from its inception in 1983 as **Silicon Valley Bancshares** to its collapse in 2023. 

From the cache, we extract SVB's hierarchy objects each quarter for the 40 years (1983-2023). For each hiearchy, we calculate a variety of complexity statistics. 

In [4]:
SVB_rssd = 1031449
asoflist = bhc_datautil.assemble_asofs(config.get('csv2sys', 'asofdate0'), config.get('csv2sys', 'asofdate1'))

In [5]:
BHCs = []
for asofdate in asoflist:
	bhcfilename = 'NIC_'+str(SVB_rssd)+'_'+str(asofdate)+'.pkl'
	bhcfilepath = os.path.join(config.get('sys2bhc', 'outdir'), bhcfilename)
	with open(bhcfilepath, 'rb') as f:
		BHCs.append(pkl.load(f))